# Silver Transformation

This notebook builds and validates the trusted Silver layer from the Bronze Delta tables.

Silver preserves the normalized TPC-H entity structure and removes Bronze-only ingestion metadata. No business aggregates or Finance measures are introduced here; those belong in Gold.

The workflow is deliberately split into two validation phases:

1. **table-local validation**: keys, required fields, source-derived Finance ranges and domains;
2. **cross-table validation**: referential integrity and Finance order-total reconciliation.

Silver tables are written only after both phases pass.

## 1. Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    BooleanType,
)

from src.config import (
    TABLES,
    BRONZE_PREFIX,
    SILVER_PREFIX,
    MARKET_SEGMENTS,
    DISCOUNT_MIN,
    DISCOUNT_MAX,
    TAX_MIN,
    TAX_MAX,
    ORDER_TOTAL_TOLERANCE,
    project_table,
)

from src.synevyr_finance_lakehouse.silver import (
    build_all_silver,
    write_all_silver,
)

from src.synevyr_finance_lakehouse.validation import (
    table_local_validations,
    referential_validations,
    order_total_reconciliation,
    net_revenue_total,
    assert_all_passed,
)

print("Silver targets:")
for table in TABLES:
    print(
        f"{project_table(BRONZE_PREFIX, table)} "
        f"-> {project_table(SILVER_PREFIX, table)}"
    )

## 2. Load Bronze Tables

All Silver transformations read from Bronze rather than directly from `samples.tpch`.

In [0]:
bronze_dfs = {
    table: spark.table(
        project_table(BRONZE_PREFIX, table)
    )
    for table in TABLES
}

## 3. Build Silver DataFrames

All eight entities are built together.

At this stage the transformations are intentionally light: the original relational columns are selected and Bronze ingestion metadata is removed.

In [0]:
silver_dfs = build_all_silver(bronze_dfs)

for table in TABLES:
    print(
        table,
        "->",
        project_table(SILVER_PREFIX, table),
    )

## 4. Validation Phase 1 - Table-Local Rules

This phase checks:

- candidate primary/composite key uniqueness;
- non-null Finance-critical and relational key fields;
- the profiled market-segment domain;
- discount and tax ranges derived during Phase 1;
- positive order totals, quantity, extended price, and retail price.

In [0]:
phase_1_results = table_local_validations(
    silver_dfs,
    market_segments=MARKET_SEGMENTS,
    discount_min=DISCOUNT_MIN,
    discount_max=DISCOUNT_MAX,
    tax_min=TAX_MIN,
    tax_max=TAX_MAX,
)

validation_schema = StructType([
    StructField("check_name", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("failed_rows", LongType(), False),
    StructField("passed", BooleanType(), False),
    StructField("details", StringType(), False),
])

phase_1_df = spark.createDataFrame(
    [result.as_tuple() for result in phase_1_results],
    schema=validation_schema,
)

display(phase_1_df.orderBy("table_name", "check_name"))

assert_all_passed(phase_1_results)
print("Validation Phase 1 passed.")

## 5. Validation Phase 2 - Cross-Table and Finance Rules

This phase verifies the TPC-H relational structure and the Finance-specific order-total rule.

Referential checks include:

- orders → customer;
- lineitem → orders, part, supplier;
- customer/supplier → nation → region;
- partsupp → part and supplier;
- composite `lineitem(part, supplier) → partsupp`.

The Finance reconciliation reproduces the TPC-H cent-level truncation semantics and applies the required `$0.01` tolerance.

In [0]:
phase_2_results = referential_validations(
    silver_dfs
)

reconciliation_result, order_reconciliation = (
    order_total_reconciliation(
        silver_dfs["orders"],
        silver_dfs["lineitem"],
        tolerance=ORDER_TOTAL_TOLERANCE,
    )
)

phase_2_results.append(reconciliation_result)

phase_2_df = spark.createDataFrame(
    [result.as_tuple() for result in phase_2_results],
    schema=validation_schema,
)

display(phase_2_df.orderBy("table_name", "check_name"))

display(
    order_reconciliation
    .orderBy(F.desc("difference"))
    .limit(20)
)

assert_all_passed(phase_2_results)
print("Validation Phase 2 passed.")

## 6. Bronze → Silver Financial Preservation

Before writing, verify that the canonical Finance net-revenue total is unchanged by the Silver transformation.

Net revenue is defined as:

`l_extendedprice × (1 - l_discount)`

Tax is intentionally excluded from this canonical Finance measure.

In [0]:
bronze_net_revenue = net_revenue_total(
    bronze_dfs["lineitem"]
)

silver_net_revenue = net_revenue_total(
    silver_dfs["lineitem"]
)

print("Bronze net revenue:", bronze_net_revenue)
print("Silver net revenue:", silver_net_revenue)
print(
    "Difference:",
    silver_net_revenue - bronze_net_revenue,
)

if silver_net_revenue != bronze_net_revenue:
    raise ValueError(
        "Bronze and Silver net revenue do not reconcile."
    )

## 7. Write Silver Tables

The Silver tables are written only after all pre-write validation has passed.

In [0]:
write_all_silver(
    silver_dfs,
    target_name=lambda table: project_table(
        SILVER_PREFIX,
        table,
    ),
)

print("Silver write complete.")

## 8. Post-Write Verification

Verify that all eight Silver tables exist and that their row counts match the validated in-memory DataFrames.

In [0]:
display(
    spark.sql(
        "SHOW TABLES IN workspace.default "
        "LIKE 'finance_silver_*'"
    )
)

row_count_checks = []

for table in TABLES:
    expected_count = silver_dfs[table].count()
    written_count = spark.table(
        project_table(SILVER_PREFIX, table)
    ).count()

    row_count_checks.append(
        (
            table,
            expected_count,
            written_count,
            expected_count - written_count,
        )
    )

row_count_schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("expected_count", LongType(), False),
    StructField("written_count", LongType(), False),
    StructField("difference", LongType(), False),
])

row_count_checks_df = spark.createDataFrame(
    row_count_checks,
    schema=row_count_schema,
)

display(
    row_count_checks_df.orderBy("table_name")
)

## 9. Silver Conclusion

If this notebook completes successfully:

- all eight normalized TPC-H entities have been built in Silver;
- Bronze-only metadata has been removed;
- candidate keys and required Finance-critical fields pass validation;
- source-derived discount, tax, positivity, and market-segment rules pass;
- all tested foreign-key relationships are valid, including the composite `(part, supplier)` relationship;
- every order reconciles to its line items within the required `$0.01` tolerance using TPC-H truncation semantics;
- canonical net revenue is unchanged from Bronze to Silver;
- written Silver row counts match the validated in-memory DataFrames.

The Silver layer is therefore ready to serve as the trusted relational source for Gold Finance analytics.

> The Finance requirement to reconcile monetary totals across **Bronze, Silver, and Gold** is only partially complete at this stage. Bronze and Silver are reconciled here; Gold must be added to the same reconciliation once the Gold layer is implemented.